# Holdout PQ of a probability-averaging ensemble of crop U-Nets (no retraining)

Import this notebook into Kaggle and press **Save & Run All**. None of the cells asks for input. It scores four configurations on the 142-file date holdout, all with flip TTA at threshold 0.45, `close_size` 11, `min_area` 400 and the adopted object filter (`filament.components.ADOPTED_OBJECT_FILTER`), the setting of the best leaderboard submission:

| config | models |
| --- | --- |
| `r34_tta` | ResNet-34 crops, 80 epochs, alone |
| `r34_v2_agree_tta` | D2 (recipe v2 with agreement weights), alone |
| `r34+agree_50_50_tta` | both, probabilities averaged 0.5 / 0.5 |
| `r34+agree_70_30_tta` | both, 0.7 (r34) / 0.3 (D2) |

The configurations come from `filament.crop_ensemble.HOLDOUT_ENSEMBLE_CONFIGS`. Each model predicts each frame once per flip, and every configuration is built from those maps, so the ensembles cost no extra network pass. Each row of `/kaggle/working/holdout_crop_ensemble.csv` has the pooled per-view PQ and the paired bootstrap difference from `r34_tta` over the same files. Choose an ensemble by that difference and its interval, by the roadmap's rule (a gain of at least 0.005 in at least 90% of resamples), not by the point PQ. Four configurations on one holdout is a small selection, and the object filter was chosen on `r34` alone, which favours that row slightly.

The `r34_tta` row is a cross-check: the B1 sweep scored that setting at 0.4255, from a float16 cache of the same maps (`docs/RESULTS.md`). It should come out very close [VERIFY].

It does not train, does not read the test images, does not write `submission.csv` and does not submit to Kaggle. Do not commit the CSV.

**Attach exactly these Kaggle inputs**

1. The competition `filament-segmentation-2026` (train images and train JSON).
2. The dataset `ignacioaguirrecandia/solar-code` (the repo zip), a version that contains `filament.crop_ensemble`. An older version stops at the imports.
3. The output of the notebook `train-unet-r34-crops-80ep` (`unet_r34_crops.pt`).
4. The output of the notebook `train-unet-r34-v2-agree` (`unet_r34_v2_agree.pt`).

**Settings**

- Accelerator: **GPU T4**, a single T4.
- Internet: **On**, for the `pip install` of `segmentation-models-pytorch==0.6.0`.

Eight tiled passes per frame (two models, four flips): about twice the time of a single-model TTA evaluation. Not measured [VERIFY]; a progress line is printed every 10 frames.

## 1. `src` on the path

`src` is the directory that contains `filament/crop_training.py`. A checkout in the working directory is used as it is. Otherwise the walk goes through `/kaggle/input` at any depth, skips every `competitions` directory, and takes the shallowest match. It is used in place and not copied.

In [ ]:
import os
import sys
from pathlib import Path


def _local_src() -> Path | None:
    """A checkout that already has the crop modules (a run off Kaggle)."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        src = candidate / "src"
        if (src / "filament" / "crop_training.py").is_file():
            return src
    return None


def _private_dataset_src(root: Path | None = None) -> Path | None:
    """Return the shallowest private-dataset ``src`` under ``root``.

    ``filament/crop_training.py`` may sit at any depth. Directories named
    ``competitions`` are not walked, so the MAGFiLO tree is left alone.
    """
    root = Path("/kaggle/input") if root is None else root
    if not root.is_dir():
        return None
    marker = Path("filament") / "crop_training.py"
    matches: list[Path] = []
    for dirpath, dirnames, _filenames in os.walk(root, topdown=True):
        dirnames[:] = [name for name in dirnames if name != "competitions"]
        current = Path(dirpath)
        if current.name != "src" or not (current / marker).is_file():
            continue
        matches.append(current)
        dirnames.clear()
    if not matches:
        return None
    return min(
        matches,
        key=lambda path: (len(path.relative_to(root).parts), path.as_posix()),
    )


def _put_on_path(src: Path) -> None:
    text = str(src)
    if text not in sys.path:
        sys.path.insert(0, text)
    current = os.environ.get("PYTHONPATH", "")
    parts = [part for part in current.split(os.pathsep) if part]
    if text not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([text, *parts]) if parts else text


def _prepare_src() -> Path:
    """Put the repo's ``src`` on ``sys.path`` and return it.

    A checkout in the working directory is used as it is. Otherwise the
    private dataset's ``src`` is used in place. It is not copied to
    ``/kaggle/working``: a copy there would be saved with this notebook's
    output and found again by a later notebook that attaches that output.
    """
    local = _local_src()
    if local is not None:
        _put_on_path(local)
        return local
    source = _private_dataset_src()
    if source is None:
        raise FileNotFoundError(
            "Could not find src/filament/crop_training.py in the working "
            "directory or under /kaggle/input (any depth outside "
            "competitions). Attach the dataset ignacioaguirrecandia/solar-code, "
            "for example "
            "/kaggle/input/datasets/ignacioaguirrecandia/solar-code/"
            "Solar-Filament-Segmentation-main/src. If it is attached, its "
            "version predates the crop modules: upload a new version of the "
            "dataset from the branch that has them. Do not make the GitHub "
            "repository public."
        )
    _put_on_path(source)
    return source


SRC = _prepare_src()
print(f"src={SRC}")
PYTHON = sys.executable

## 2. Install the segmentation library

`segmentation-models-pytorch` **0.6.0** (MIT), the version that trained the checkpoints. Kaggle already has PyTorch; this cell does not install it.

In [ ]:
import subprocess

# Pinned. segmentation-models-pytorch 0.6.0 is MIT.
SMP_VERSION = "0.6.0"

completed = subprocess.run(
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        f"segmentation-models-pytorch=={SMP_VERSION}",
    ],
    check=False,
)
if completed.returncode != 0:
    raise RuntimeError(
        f"segmentation-models-pytorch=={SMP_VERSION} did not install. Stop. "
        "Internet has to be On. Kaggle already has PyTorch; this cell only "
        "installs segmentation-models-pytorch (MIT)."
    )

In [ ]:
import segmentation_models_pytorch as smp

print(f"segmentation-models-pytorch {smp.__version__}")
if smp.__version__ != SMP_VERSION:
    raise RuntimeError(
        f"Expected segmentation-models-pytorch {SMP_VERSION}, "
        f"found {smp.__version__}. Stop."
    )

## 3. Settings and the weights

`MEMBERS` are the models the configurations need. Each loads from its entry's `weights_path` in `filament.checkpoint` (`WEIGHTS_PATHS` overrides one by name) and goes through the checkpoint guard against its own entry. Every member has to have a pinned `tensor_sha256`; leave `ALLOW_UNVERIFIED_WEIGHTS` as `False`. Tile overlap and batch are the training notebook's values.

In [ ]:
import zipfile

from filament.checkpoint import crop_checkpoint
from filament.components import ADOPTED_OBJECT_FILTER, object_cut_from_config
from filament.crop_ensemble import ENSEMBLE_POINT, HOLDOUT_ENSEMBLE_CONFIGS
from filament.ensemble import needed_flips

CONFIGS = HOLDOUT_ENSEMBLE_CONFIGS
POINT = ENSEMBLE_POINT
OBJECT_CUT = object_cut_from_config(ADOPTED_OBJECT_FILTER)
MEMBERS = tuple(needed_flips(CONFIGS))
WEIGHTS_PATHS = {name: crop_checkpoint(name).weights_path for name in MEMBERS}
ALLOW_UNVERIFIED_WEIGHTS = False

TILE_OVERLAP = 128
INFER_BATCH_SIZE = 8
USE_AMP = True
SEED = 0

OUTPUT_PATH = Path("/kaggle/working/holdout_crop_ensemble.csv")
KAGGLE_WORKING = Path("/kaggle/working")
print(f"members={MEMBERS} point={POINT} object_filter={OBJECT_CUT}")


def _archive_error(path: Path, reason: str) -> RuntimeError:
    """Explain a Kaggle-extracted ``.pt`` that cannot be re-zipped."""
    return RuntimeError(
        f"{path} is a directory, not a checkpoint file. "
        "Kaggle extracts an uploaded .pt because torch.save writes a zip, "
        "so the file becomes a folder such as unet.pt/unet/data.pkl. "
        f"This notebook did not re-zip it: {reason} "
        "Set WEIGHTS_PATH to a real .pt file that Kaggle did not extract."
    )


def _rezip_extracted_archive(directory: Path, dest: Path | None = None) -> Path:
    """Re-zip one extracted torch archive to a file under working.

    Members keep the archive folder name (``unet/data.pkl``). The zip is
    stored, not deflated, so ``torch.load`` can read it. The directory
    itself is never returned.
    """
    if not directory.is_dir():
        raise _archive_error(directory, "the path is not a directory.")
    pkls = sorted(
        path
        for path in directory.rglob("data.pkl")
        if path.is_file() and not path.is_symlink()
    )
    if len(pkls) != 1:
        raise _archive_error(
            directory,
            f"expected one data.pkl, found {len(pkls)}.",
        )
    archive_dir = pkls[0].parent
    if archive_dir.parent == archive_dir:
        raise _archive_error(directory, "data.pkl has no parent folder.")
    names = {child.name for child in archive_dir.iterdir()}
    storage = (archive_dir / "data").is_dir()
    marked = "version" in names or "byteorder" in names or storage
    if "data.pkl" not in names or not marked:
        raise _archive_error(
            directory,
            "data.pkl is not next to version, byteorder, or a data folder.",
        )
    files: list[Path] = []
    ordered = sorted(archive_dir.rglob("*"), key=lambda item: item.as_posix())
    for path in ordered:
        if path.is_symlink():
            raise _archive_error(directory, f"{path.name} is a symlink.")
        if path.is_file():
            files.append(path)
    if not files:
        raise _archive_error(directory, "the archive folder has no files.")
    try:
        KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
    except OSError as exc:
        raise _archive_error(
            directory,
            f"/kaggle/working is not writable ({exc}).",
        ) from exc
    stem = directory.stem if directory.name.endswith(".pt") else directory.name
    if dest is None:
        dest = KAGGLE_WORKING / f"{stem}-from-extracted.pt"
    try:
        dest.resolve().relative_to(archive_dir.resolve())
    except ValueError:
        inside = False
    else:
        inside = True
    if inside:
        raise _archive_error(
            directory,
            "the re-zipped file would land inside the extracted archive.",
        )
    with zipfile.ZipFile(dest, "w", compression=zipfile.ZIP_STORED) as handle:
        for file in files:
            rel = file.relative_to(archive_dir.parent).as_posix()
            if rel.startswith("/") or ".." in rel.split("/"):
                raise _archive_error(directory, f"unsafe member {rel}.")
            handle.write(file, arcname=rel)
    print(f"Kaggle extracted the torch archive at {directory}. Re-zipped it to {dest}.")
    return dest


def _materialize_checkpoint(path: Path) -> Path:
    """Return a real ``.pt`` file. Re-zip an extracted archive."""
    if path.is_file():
        return path
    if path.is_dir():
        return _rezip_extracted_archive(path)
    raise FileNotFoundError(
        f"{path} does not exist. Set WEIGHTS_PATH to an existing .pt file."
    )

## 4. Load every member and check it

Each checkpoint prints its stored fields and `tensor_sha256` and goes through the guard against its own entry. The notebook stops if a member is missing, unpinned or not the expected file, or if there is no CUDA GPU.

In [ ]:
import random
import time

import numpy as np
import torch

from filament import crop_training as ct
from filament.checkpoint import checkpoint_fingerprint, verify_and_report
from filament.crop_ensemble import require_pinned

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = ct.require_cuda(6.0)
print(f"device={torch.cuda.get_device_name(0)} torch={torch.__version__}")

require_pinned(MEMBERS, allow_unverified=ALLOW_UNVERIFIED_WEIGHTS)

PREDICTORS = {}
for name in MEMBERS:
    expected = crop_checkpoint(name)
    path = Path(WEIGHTS_PATHS[name])
    if not path.exists():
        raise FileNotFoundError(
            f"{name}: {path} does not exist. Attach the output of the notebook "
            f"{Path(expected.weights_path).parent.name}, or set WEIGHTS_PATHS."
        )
    weights_file = _materialize_checkpoint(path)
    checkpoint = ct.load_checkpoint(weights_file)
    fingerprint = checkpoint_fingerprint(checkpoint)
    print(f"{name}: weights={weights_file}")
    for key in ("epochs", "status", "encoder_name", "crop_size", "smp_version"):
        print(f"  {key}={checkpoint.get(key, 'not stored')}")
    print(f"  tensor_sha256={fingerprint['tensor_sha256']}")
    verify_and_report(
        fingerprint,
        allow_unverified=ALLOW_UNVERIFIED_WEIGHTS,
        expected=expected,
    )
    tile = int(checkpoint.get("tile_size") or checkpoint["crop_size"])
    PREDICTORS[name] = ct.tiled_predictor(
        ct.model_from_checkpoint(checkpoint, device),
        device,
        tile_size=tile,
        overlap=TILE_OVERLAP,
        batch_size=INFER_BATCH_SIZE,
        use_amp=USE_AMP,
    )
    del checkpoint

## 5. Date holdout

The same split as the training notebook: the most recent 20% of calendar dates (142 files with 231 annotator views on the current data). Only the train images and the train JSON are read.

In [ ]:
from filament.annotations import (
    TRAIN_ANNOTATIONS_PATH,
    TRAIN_IMAGES_DIR,
    load_train_annotations,
)
from filament.sweep import count_views
from filament.train_data import split_training_images

IMAGE_DIR = Path(TRAIN_IMAGES_DIR)
ANNOTATIONS_PATH = Path(TRAIN_ANNOTATIONS_PATH)
if not ANNOTATIONS_PATH.is_file() or not IMAGE_DIR.is_dir():
    raise FileNotFoundError(
        "Train data was not found. Checked "
        f"images={IMAGE_DIR} and annotations={ANNOTATIONS_PATH}. "
        "Attach the competition filament-segmentation-2026 to this notebook."
    )
holdout = split_training_images(load_train_annotations(ANNOTATIONS_PATH).images).holdout
holdout_views = count_views(holdout)
print(f"holdout files={len(holdout)} annotator views={holdout_views}")
if (len(holdout), holdout_views) != (142, 231):
    print(
        "WARNING: the holdout is not 142 files with 231 views, the numbers the "
        "training notebook used. [VERIFY] the data version."
    )
print("test/test_images is not used")

## 6. Predict, average, score

Per frame and model, the four flips are predicted once. Each configuration averages its models' flip means with its weights, then thresholds, labels, applies the object filter and scores with the pooled per-view PQ (`filament.metrics.fast_view_counts`), kept per file for the paired bootstrap.

In [ ]:
from filament.crop_data import read_gray
from filament.crop_ensemble import (
    ensemble_rows,
    per_file_ensemble_counts,
    write_ensemble_rows,
)


def _read(image):
    return read_gray(IMAGE_DIR / image.file_name, (image.height, image.width))


def _progress(index, total, image):
    if index == 1 or index % 10 == 0 or index == total:
        print(f"  holdout {index}/{total} {image.file_name}", flush=True)


begin = time.monotonic()
per_file, inference_seconds = per_file_ensemble_counts(
    holdout,
    _read,
    PREDICTORS,
    CONFIGS,
    point=POINT,
    cut=OBJECT_CUT,
    progress=_progress,
)
rows = ensemble_rows(per_file, CONFIGS, point=POINT, cut=OBJECT_CUT)
write_ensemble_rows(OUTPUT_PATH, rows)
for row in rows:
    print(
        f"{row['config']:<22} {row['models']:<28} pq={row['pooled_pq']} "
        f"diff={row['diff_vs_first']} [{row['diff_low']}, {row['diff_high']}] "
        f"better={row['fraction_better']}"
    )
print(
    f"inference {inference_seconds:.0f} s, total {time.monotonic() - begin:.0f} s, "
    f"{inference_seconds / len(holdout):.1f} s per frame"
)
print(f"wrote {OUTPUT_PATH}. Do not commit it.")
print("This notebook wrote no submission file and did not submit to Kaggle.")